# Autonomous AI Data Analyst — Colab Setup, Backup & GitHub Push

**This notebook works with the downloadable project ZIP.** Nothing in this notebook contains a real Gemini API key or GitHub token. Run cells from top to bottom. The Streamlit web interface should be hosted externally rather than inside a Colab notebook.

## 1. Upload your ZIP and extract it

Run this cell, choose `Autonomous_AI_Data_Analyst_Safe_Project.zip`, then let the notebook extract its safe project files.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_names:
    raise ValueError("Please upload the project ZIP")
project_dir = Path("/content/autonomous-ai-data-analyst-safe")
project_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    for info in archive.infolist():
        relative = Path(info.filename)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Unsafe ZIP path")
        if not info.is_dir():
            target = (Path("/content") / relative).resolve()
            if not target.is_relative_to(Path("/content")):
                raise ValueError("Unsafe extraction path")
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
print("Extracted project:", project_dir)
print("Files:", sorted(p.name for p in project_dir.iterdir()))

## 2. Install dependencies

Installs the Python libraries used by the project. The web app itself should be deployed on Streamlit Community Cloud, Hugging Face Spaces, or another Python hosting service.

In [ ]:
%pip install -q -r /content/autonomous-ai-data-analyst-safe/requirements.txt

## 3. Try the analytical tools in Colab

Upload a **non-sensitive** CSV/XLSX file. This step uses Pandas locally and does not call Gemini.

In [ ]:
import sys
import pandas as pd
from pathlib import Path
sys.path.insert(0, str(project_dir))
from analyst_engine import columns_profile, run_tool
from google.colab import files

sample_upload = files.upload()
filename = next(iter(sample_upload))
if filename.lower().endswith(".csv"):
    dataset = pd.read_csv(filename)
elif filename.lower().endswith(".xlsx"):
    dataset = pd.read_excel(filename)
else:
    raise ValueError("Only CSV or XLSX allowed")
print("Profile:", columns_profile(dataset))
print("Quality:", run_tool(dataset, "data_quality", {}))

## 4. Optional: Ask Gemini a question

Your key is entered through a hidden password prompt. Do **not** paste it into a source cell or print it. The project model must be enabled for your Gemini API account.

In [ ]:
from getpass import getpass
from google import genai
from analyst_engine import answer_question

secret_key = getpass("Gemini API key (hidden): ")
client = genai.Client(api_key=secret_key)
question = input("Ask a question about the uploaded data: ").strip()
if question:
    outcome = answer_question(dataset, client, question)
    print(outcome["answer"])
else:
    print("No question entered")

## 5. Save a backup to your Google Drive

You must authorize Google Drive access in Colab. The cell copies the **secret-free ZIP** into your Drive; it does not upload it automatically without your authorization.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")
backup_dir = Path("/content/drive/MyDrive/AI_Data_Analyst_Backups")
backup_dir.mkdir(parents=True, exist_ok=True)
source_zip = Path("/content") / zip_names[0]
target = backup_dir / source_zip.name
shutil.copy2(source_zip, target)
print("Backup saved to:", target)

## 6. Optional: Push safe project files to GitHub

**Caution:** This modifies your repository. Use a **fine-grained GitHub personal access token** restricted to your project repository, granting `Contents: Read and write`. Enter it through the hidden prompt. The notebook uses a temporary `GIT_ASKPASS` script and does not print or save the token into your repository. Revoke the token after pushing.

This pushes code files, not your original dataset, secrets, or ZIP. It does not delete existing notebook files. If the repository has conflicting changes, the command stops rather than overwriting them.

In [ ]:
from getpass import getpass
import os, shutil, subprocess, tempfile
from pathlib import Path

repository = "Ashiwany25/autonomous-ai-data-analyst"
commit_email = input("Public/commit GitHub email (use GitHub noreply if preferred): ").strip()
commit_name = input("GitHub commit display name: ").strip() or "Project Contributor"
if not commit_email:
    raise ValueError("Please enter a Git commit email")
token = getpass("Fine-grained GitHub token (hidden): ")
if not token:
    raise ValueError("Token is required")

# Temporary helper prints credentials only to Git, never to the notebook output.
with tempfile.TemporaryDirectory() as tmp:
    helper = Path(tmp) / "askpass.sh"
    helper.write_text('#!/bin/sh\ncase "$1" in *Username*) printf "%s" "x-access-token";; *) printf "%s" "$MY_GH_TOKEN";; esac\n')
    helper.chmod(0o700)
    env = os.environ.copy()
    env.update({"GIT_ASKPASS": str(helper), "GIT_TERMINAL_PROMPT": "0", "MY_GH_TOKEN": token})
    checkout = Path(tmp) / "checkout"
    def git(*args, cwd=None):
        subprocess.run(["git", *args], cwd=cwd, env=env, check=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    git("clone", "--branch", "main", f"https://github.com/{repository}.git", str(checkout))
    # Copy only known-safe source/config files, without touching the original notebook.
    paths = ["app.py", "analyst_engine.py", "requirements.txt", "README.md", ".gitignore",
             ".dockerignore", ".env.example", "Dockerfile", ".streamlit/config.toml",
             "tests/test_tools.py", "COLAB_SETUP_AND_BACKUP.ipynb"]
    for rel in paths:
        src = project_dir / rel
        dst = checkout / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    git("config", "user.name", commit_name, cwd=checkout)
    git("config", "user.email", commit_email, cwd=checkout)
    git("add", "--", *paths, cwd=checkout)
    diff = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=checkout)
    if diff.returncode == 0:
        print("No changes to commit")
    else:
        git("commit", "-m", "Add portable, secret-free analyst project files", cwd=checkout)
        git("push", "origin", "main", cwd=checkout)
        print("Project files pushed to GitHub main branch")
    env.pop("MY_GH_TOKEN", None)
del token
print("Temporary Git credentials discarded. Revoke token after use.")